# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SUKRIT004/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: one content page for one client on one reporting day. I will use the March 2026 partition as the development slice. The goal is to use information available at a decision moment to rank pages for CTR/engagement review. The monthly partition contains daily observations, so the same page can appear on multiple reporting dates.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from huggingface_hub import hf_hub_download
import pandas as pd

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march = pd.read_parquet(march_file)

print("Shape:", march.shape)
print("\nColumns:")
print(march.columns.tolist())

print("\nFirst 5 rows:")
display(march.head())

Shape: (9841378, 30)

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

First 5 rows:


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: gsc_impressions, gsc_avg_position, ga4_sessions, ga4_engaged_sessions, and scroll_events. These are candidate signals that can be available at the time of prioritization, subject to the relevant availability flags.

Label/proxy: a future position-adjusted CTR opportunity measure derived from later search performance. I will keep the future outcome separate from the features so it cannot leak into the decision-time inputs.

Context: report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4, gsc_data_available, and ga4_data_available.

Excluded: future performance fields and any label-derived variables from the feature set. I also exclude fields that are not available at the decision moment because using them would create leakage.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-04 — Three verification queries

# Query 1 — Grain
key_counts = (
    march
    .groupby(["report_date", "client_hash_id", "content_hash_id"])
    .size()
)

print("=== Query 1: Grain ===")
print("Rows:", len(march))
print("Unique date/client/content combinations:", key_counts.shape[0])
print("Maximum rows per date/client/content key:", key_counts.max())
print("Duplicate keys:", (key_counts > 1).sum())


# Query 2 — Row count and date span
print("\n=== Query 2: Row count and date span ===")
print("March row count:", len(march))
print("Minimum report_date:", march["report_date"].min())
print("Maximum report_date:", march["report_date"].max())
print("Unique report dates:", march["report_date"].nunique())


# Query 3 — Availability
# Pandas equivalent of:
# WHERE gsc_data_available IS TRUE
#   AND ga4_data_available IS TRUE

available = march[
    march["gsc_data_available"].eq(True) &
    march["ga4_data_available"].eq(True)
]

print("\n=== Query 3: Availability ===")
print("Rows where GSC and GA4 are both TRUE:", len(available))
print("Rows surviving availability filter:", len(available))

# Five-feature frame
feature_cols = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "scroll_events"
]

feature_frame = available[
    ["report_date", "client_hash_id", "content_hash_id"] + feature_cols
].copy()

print("\n=== Five-feature frame ===")
print("Shape:", feature_frame.shape)
display(feature_frame.head())

# Deliberate leakage experiment
# Create a label-derived column using current-period clicks and impressions.
# This is intentionally WRONG and demonstrates leakage.

available = available.copy()

available["leaky_ctr"] = (
    available["gsc_clicks"] /
    available["gsc_impressions"].replace(0, pd.NA)
)

print("\n=== Leakage trap ===")
print("Leaky feature created: leaky_ctr")
print("This column is derived from the outcome we are trying to rank.")
print("It must NOT be included in the honest feature frame.")

available = available.drop(columns=["leaky_ctr"])

print("Leaky feature removed.")



=== Query 1: Grain ===
Rows: 9841378
Unique date/client/content combinations: 9841378
Maximum rows per date/client/content key: 1
Duplicate keys: 0

=== Query 2: Row count and date span ===
March row count: 9841378
Minimum report_date: 2026-03-01
Maximum report_date: 2026-03-31
Unique report dates: 31

=== Query 3: Availability ===
Rows where GSC and GA4 are both TRUE: 364347
Rows surviving availability filter: 364347

=== Five-feature frame ===
Shape: (364347, 8)


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,scroll_events
14480,2026-03-01,client_65de48885f4ef01b,content_5c80451459c29b4a,5,5.400000,1.0,0.0,0.0
14748,2026-03-01,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,39,5.666667,2.0,0.0,0.0
14768,2026-03-01,client_65de48885f4ef01b,content_e25ea7297a1dffd3,179,5.156425,2.0,0.0,0.0
14783,2026-03-01,client_65de48885f4ef01b,content_6b0149a80607dac3,72,7.694444,1.0,0.0,0.0
14855,2026-03-01,client_65de48885f4ef01b,content_62673eea26c31c17,3282,6.167885,1.0,0.0,0.0



=== Leakage trap ===
Leaky feature created: leaky_ctr
This column is derived from the outcome we are trying to rank.
It must NOT be included in the honest feature frame.
Leaky feature removed.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can support observational and directional analysis, but it cannot establish causal effects. History may be unbalanced across pages and clients, so some pages have more observations than others. Some periods may have GSC data available without GA4 data, so combining signals requires availability checks. Rolling or multi-day metrics can also overlap across monthly snapshots, meaning observations from different months should not automatically be treated as independent. The final June 2026 month should be treated as a sealed outcome/test period rather than used to develop label logic.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.